# Develop and test a quantum error-correction scheme with `qdk.ec`

A code definition, a circuit, and the meaning of its measurements are different parts of a design. Changing one can break the others. This walkthrough makes those changes deliberately: compare two implementations, repair readout equations, and follow faults through a syndrome-extraction circuit.

`qodec` holds the declarations and saves them. `qdk.ec` computes and checks their consequences:

| Workflow | API | Question |
| --- | --- | --- |
| Analyze | `ec.SubsystemCode`, `ec.GadgetProfile` | What does this code or circuit do, and how do specified faults propagate? |
| Audit | `ec.audit` | Which declarations disagree or leave something unspecified? |
| Derive | `ec.derive` | Which checks and observable bindings can exact simulation supply? |
| Synthesize | `ec.build_qodec` | Which gadget circuits can we construct from a code definition? |

## Installing

`qdk.ec` is an optional extra of the `qdk` package:

```bash
pip install "qdk[ec]"
```

## 1. Load a qodec

Start from `c4.qodec.yaml`, next to this notebook. It describes the $[[4,2,2]]$ code: four physical qubits encode two logical qubits, with distance two. With ideal syndrome measurements, it detects arbitrary single-qubit data errors. Whether a circuit fault is detected is a separate question, explored below.

In [14]:
from collections import Counter
from pathlib import Path

from IPython.display import Markdown, display
import qodec as qc
import qdk.ec as ec

protocol = qc.Qodec.load("c4.qodec.yaml")
print(protocol.description)

Protocol based on Knill's C4


A qodec is a chain of **layers**, from the most abstract instruction set down to
the most concrete. Each layer carries the **gadgets** that lower one of its
instructions into a circuit over the layer below. Here there is a single lowering
edge: the logical `C4` instruction set down to physical `stim` operations.

In [2]:
layer = protocol.layers[0]
print("lowering:", layer.instruction_set.name, "->", protocol.layers[1].instruction_set.name)
print("gadgets: ", sorted(layer.gadgets))

lowering: C4 -> stim
gadgets:  ['idle', 'measure_xx', 'measure_zz', 'prepare_xx', 'prepare_zz', 'transversal_cx', 'x0', 'x1', 'z0', 'z1']


## 2. Profile the code and gadgets

`SubsystemCode` adds algebraic analysis to qodec's code data. `GadgetProfile` reports facts obtained through exact simulation.

In [13]:
code = ec.SubsystemCode.of(protocol.codes["C4"])

print("stabilizers:", list(code.stabilizers))
print("logical basis:", list(code.logical_basis))

distance, witness = code.distance()
logical_error = ec.Pauli.identity()
for factor in witness:
    logical_error *= factor

print("distance:", distance)
print("witness factors:", [str(factor) for factor in witness])
print("combined error:", logical_error)
print("weight:", logical_error.weight)
print("syndrome:", sorted(code.syndrome_of(logical_error)))
print("logical effect:", code.logical_effect_of(logical_error))

assert distance == logical_error.weight == 2
assert not code.syndrome_of(logical_error)
assert code.logical_effect_of(logical_error).weight > 0

stabilizers: [XXXX, ZZZZ]
logical basis: [XX, ZIZ, XIX, ZZ]
distance: 2
witness factors: ['X', 'IX']
combined error: XX
weight: 2
syndrome: []
logical effect: X


The returned witness is a list of Pauli factors. Their product has weight two, an empty syndrome, and a nontrivial logical effect: it changes encoded information without violating a stabilizer. That is a concrete distance-two witness.

The code cannot guarantee correction of every arbitrary single-qubit error. This does not rule out correction with extra information, nor does code distance alone certify a measurement circuit.

### Declared vs. realized action

A gadget's implemented instruction states what it should do; its circuit is an independent implementation of that instruction. `GadgetProfile.objective` describes the instruction, while `GadgetProfile.action` describes the circuit. Comparing them can catch a transcription error without treating the circuit itself as the specification.

In [ ]:
measure_zz = layer.gadgets["measure_zz"]
profile = ec.GadgetProfile(measure_zz)
objective = profile.objective
assert objective is not None

print("objective:", objective)
print("action:   ", profile.action)
print("mismatch: ", profile.action.why_not_equivalent_to(objective) or "none")

objective: observables: FrameGroup(generators=(Z^{2}, IZ^{3}))
stabilizers: FrameGroup(generators=())
mapping: {}
action:    observables: FrameGroup(generators=(Z^{5,6,8}, IZ^{5,6,7}))
stabilizers: FrameGroup(generators=())
mapping: {}
mismatch:  none


### Checks and readouts

A circuit emits measurement bits. Checks combine bits and encoding signs into parities that should be zero without faults. Readouts specify how those measurements give the instruction's logical results or flags.

The profile below returns **measurement-record projections**: integers index `measure_zz.circuit.readouts`. These projections omit boundary-frame terms, so they are not complete qodec reference equations. Inspect the gadget's declared readouts alongside them.

Exact simulation can derive checks and observable bindings for supported circuits. Flag equations must be supplied by the author, and an audit remains necessary.

In [24]:
print("Discovered checks, measurement positions only:")
for positions in profile.checks:
    print(sorted(positions))

display(Markdown("\n".join([
    "| Readout | Profile measurement positions | Authored equation |",
    "| ---: | --- | --- |",
    *(f"| {position} | `{sorted(positions)}` | `{readout}` |"
      for position, (positions, readout) in enumerate(zip(profile.readouts, measure_zz.readouts))),
])))

Discovered checks, measurement positions only:
[0, 1, 2, 3]


| Readout | Profile measurement positions | Authored equation |
| ---: | --- | --- |
| 0 | `[0, 2]` | `["circuit.readouts[0]", "circuit.readouts[2]"]` |
| 1 | `[0, 1]` | `["circuit.readouts[0]", "circuit.readouts[1]"]` |

## 3. Audit and repair the readout equations

The circuit can implement the right logical measurement while its declared readout equation reports the wrong answer. The authored `measure_xx` and `measure_zz` readouts omit the incoming logical-frame signs: their parities work for a zero frame, but not for an arbitrary known Pauli correction.

`ec.audit` returns errors, warnings, and informational findings without changing the protocol. First summarize them by rule and inspect one error. Its filename and one-based line locate the loaded declaration; the layer and equation indices are zero-based.

In [15]:
report = ec.audit(protocol)
counts = Counter((item.severity.name, item.rule) for item in report.diagnostics)
display(Markdown("\n".join([
    "| Severity | Rule | Count |",
    "| --- | --- | ---: |",
    *(f"| {severity} | `{rule}` | {count} |"
      for (severity, rule), count in sorted(counts.items())),
])))

readout_error = next(item for item in report.errors if item.rule == "gadget/readout-mismatch")
print(ec.Report((readout_error,)))

readout_node = protocol.resolve('layers[0].gadgets["measure_xx"].readouts[0].equation')
assert readout_node.source_location is not None
assert readout_error.source_location is not None
assert readout_node.source_location.path == readout_error.source_location.path
assert readout_node.source_location.line == readout_error.source_location.line

| Severity | Rule | Count |
| --- | --- | ---: |
| ERROR | `gadget/readout-mismatch` | 4 |
| WARNING | `gadget/incomplete-output-frame` | 12 |

[ERROR] gadget/readout-mismatch
~/repositories/qdk/samples/notebooks/qdk_ec/c4.qodec.yaml:241
layers[0].gadgets['measure_xx'] (C4 -> stim)
readouts[0] does not report the required logical X_0 measurement
    Declared equation: ["circuit.readouts[0]", "circuit.readouts[1]"]
    Verified readout equation: ["in[0].x[0]", "circuit.readouts[0]", "circuit.readouts[1]"]

audit: 1 error(s), 0 warning(s), 0 informational


### Apply the missing frame signs

For each logical X measurement, include the corresponding `in[0].x[...]` sign; for logical Z, include `in[0].z[...]`. The verified equation above shows the first correction. Apply the same reasoning to both outputs of each measurement gadget on a fresh copy, without parsing a diagnostic's prose into code.

In [16]:
repaired = qc.Qodec.load("c4.qodec.yaml")
for mnemonic, basis in (("measure_xx", "x"), ("measure_zz", "z")):
    measurement = repaired.layers[0].gadgets[mnemonic]
    measurement.readouts = [
        [*readout.equation, f"in[0].{basis}[{position}]"]
        for position, readout in enumerate(measurement.readouts)
    ]

repaired_report = ec.audit(repaired)
display(Markdown("\n".join([
    "| Version | Errors | Warnings | INFO | report.ok |",
    "| --- | ---: | ---: | ---: | --- |",
    *(f"| {name} | {len(result.errors)} | {len(result.warnings)} | "
      f"{len(result.informational)} | {result.ok} |"
      for name, result in (("Authored", report), ("Frame signs added", repaired_report))),
])))

assert repaired_report.ok
assert len(repaired_report.warnings) == 12
assert len(ec.audit(protocol).errors) == 4

| Version | Errors | Warnings | INFO | report.ok |
| --- | ---: | ---: | ---: | --- |
| Authored | 4 | 12 | 0 | False |
| Frame signs added | 0 | 12 | 0 | True |

The four readout errors are gone. The twelve warnings still identify output stabilizer signs not determined by the declared relations. `report.ok` means **no errors**, not that every concern is resolved or that the protocol is fault tolerant.

## 4. Derive missing declarations

Now remove checks from a separate gadget draft and ask `ec.derive` to supply them. Keep its authored readouts so the experiment isolates missing checks. Derivation returns a new artifact; it is not a replacement for audit, and it does not infer flag equations.

In [17]:
draft = qc.Gadget(
    measure_zz.implements,
    measure_zz.circuit,
    inputs=measure_zz.inputs,
    outputs=measure_zz.outputs,
    checks=[],
    readouts=measure_zz.readouts,
    parameter_bindings=measure_zz.parameter_bindings,
    metadata=measure_zz.metadata,
)
completed = ec.derive(draft)
assert isinstance(completed, qc.Gadget)
assert not draft.checks and completed.checks

display(Markdown("\n".join([
    "| Version | Declared checks | Declared readouts |",
    "| --- | ---: | ---: |",
    *(f"| {name} | {len(item.checks)} | {len(item.readouts)} |"
      for name, item in (("Draft", draft), ("Derived", completed))),
])))
for check in completed.checks:
    print([str(term) for term in check])

| Version | Declared checks | Declared readouts |
| --- | ---: | ---: |
| Draft | 0 | 2 |
| Derived | 1 | 2 |

['circuit.readouts[0]', 'circuit.readouts[1]', 'circuit.readouts[2]', 'circuit.readouts[3]', 'in[0].stabilizers[1]']


### Derivation is not automatic repair

`ec.derive` can also process every gadget in a qodec. The current C4 derivation still leaves the incoming-frame omissions and incomplete output-frame relations reported by audit. Compare counts rather than assuming a completed artifact is certified.

In [18]:
completed_protocol = ec.derive(protocol)
assert isinstance(completed_protocol, qc.Qodec)
derived_report = ec.audit(completed_protocol)

display(Markdown("\n".join([
    "| Version | Errors | Warnings | INFO |",
    "| --- | ---: | ---: | ---: |",
    *(f"| {name} | {len(result.errors)} | {len(result.warnings)} | {len(result.informational)} |"
      for name, result in (("Authored", report), ("Derived", derived_report))),
])))
assert completed_protocol is not protocol
assert Counter(item.rule for item in derived_report.diagnostics) == Counter(
    item.rule for item in report.diagnostics
)

| Version | Errors | Warnings | INFO |
| --- | ---: | ---: | ---: |
| Authored | 4 | 12 | 0 |
| Derived | 4 | 12 | 0 |

## 5. Compare implementations of the same instruction

The logical `x0` gadget uses `X 0 1`. Try `X 2 3` instead, keeping its implemented instruction and encodings fixed. The two physical operators differ by the code's X stabilizer, `X_0 X_1 X_2 X_3`, so they should act identically on encoded states.

Also try `X 0`, which is not the same logical operation. Structural equality compares declarations; `GadgetProfile.is_equivalent_to` compares their realized logical actions and boundary encodings.

In [19]:
original_x = layer.gadgets["x0"]
original_profile = ec.GadgetProfile(original_x)
comparison_rows = [
    "| Circuit | Structurally equal | Logically equivalent | Explanation |",
    "| --- | --- | --- | --- |",
]
equivalence = {}
for source in (original_x.circuit.source, "X 2 3", "X 0"):
    candidate = qc.Gadget(
        original_x.implements,
        qc.gadgets.Circuit(original_x.circuit.instruction_set, source, format="stim"),
        inputs=original_x.inputs,
        outputs=original_x.outputs,
        checks=original_x.checks,
        readouts=original_x.readouts,
        parameter_bindings=original_x.parameter_bindings,
        metadata=original_x.metadata,
    )
    candidate_profile = ec.GadgetProfile(candidate)
    equivalent = original_profile.is_equivalent_to(candidate_profile)
    equivalence[source.strip()] = equivalent
    reason = original_profile.why_not_equivalent_to(candidate_profile) or "Same logical action"
    comparison_rows.append(
        f"| `{source.strip()}` | {original_x == candidate} | {equivalent} | {reason} |"
    )
display(Markdown("\n".join(comparison_rows)))
assert equivalence == {"X 0 1": True, "X 2 3": True, "X 0": False}

| Circuit | Structurally equal | Logically equivalent | Explanation |
| --- | --- | --- | --- |
| `X 0 1` | True | True | Same logical action |
| `X 2 3` | False | True | Same logical action |
| `X 0` | False | False | Logical actions differ in their outcome-dependent Pauli signs. |

## 6. Follow faults through a gadget

Noiseless equivalence does not say how an implementation responds to faults. The `idle` gadget measures the C4 stabilizers using ancillas 4 and 5. First number its parsed calls: `FaultEvent.after` uses these zero-based call positions, not source-file line numbers.

Inject an X error on the syndrome ancilla after its first coupling, then compare X and Z errors on the data qubit at the same point. `effects_of` evaluates the whole list in one simulation. `fault_effects` provides a larger, canonical list of X and Z faults after each call on its touched qubits.

In [ ]:
idle = layer.gadgets["idle"]
idle_profile = ec.GadgetProfile(idle)
calls = idle.circuit.calls
for position, call in enumerate(calls):
    print(f"{position:2d}: {call.mnemonic} {' '.join(map(str, call.operands))}")

coupling = next(
    position for position, call in enumerate(calls)
    if call.mnemonic == "CX" and call.operands == [4, 0]
)
fault_cases = [
    ("X on ancilla 4", ec.FaultEvent.after(coupling, ec.Pauli({4: "X"}))),
    ("X on data 0", ec.FaultEvent.after(coupling, ec.Pauli({0: "X"}))),
    ("Z on data 0", ec.FaultEvent.after(coupling, ec.Pauli({0: "Z"}))),
]
effects = idle_profile.effects_of([fault for _, fault in fault_cases])
fault_rows = [
    "| Fault after call | Injected error | Checks flipped | Readouts flipped | Output Pauli |",
    "| ---: | --- | --- | --- | --- |",
]
for (label, fault), effect in zip(fault_cases, effects):
    output = ", ".join(
        f"block {entry}: {error}" for entry, error in sorted(effect.output_error.items())
    ) or "none"
    fault_rows.append(
        f"| {coupling} | {label} | {sorted(effect.syndrome)} | "
        f"{sorted(effect.readout_flips)} | {output} |"
    )
display(Markdown("\n".join(fault_rows)))
assert effects[0].syndrome == {1}
assert effects[0].output_error[0].weight > 0
assert effects[2].syndrome == {2}

 0: R 4
 1: R 5
 2: H 4
 3: CX 4 0
 4: CX 4 1
 5: CX 4 2
 6: CX 4 3
 7: H 4
 8: CX 0 5
 9: CX 1 5
10: CX 2 5
11: CX 3 5
12: M 4
13: M 5


| Fault after call | Injected error | Checks flipped | Readouts flipped | Output Pauli |
| ---: | --- | --- | --- | --- |
| 3 | X on ancilla 4 | [1, 3] | [] | block 0: XX |
| 3 | X on data 0 | [1, 3] | [] | block 0: XX |
| 3 | Z on data 0 | [] | [] | block 0: ZZ |

The ancilla X fault propagates through later couplings onto the data. It and the data X fault produce the same reported effects here; different causes need not have different syndromes.

Check indices refer to `idle.checks`, readout indices to `idle.readouts`, and output entries to `idle.outputs`. Complete check equations include output-sign terms: the data Z fault flips check 2. The output Pauli records flips of encoded logical probes, not the full physical error, so it is not by itself proof that a residual error preserves the output codespace.

These are deterministic effects of specified Pauli faults. No probabilities, decoder, or fault-tolerance guarantee have been assumed. `idle_profile.distance()` searches for the fewest allowed faults whose combined effect leaves every declared check unchanged, preserves every output codespace, and changes the realized logical action; `distance_bounds()` bounds the same quantity. Indicators come from the action's prepared-state stabilizers, preserved logical mappings, and logical measurement signs. A logical Z fault on a prepared logical zero is harmless. Changes to measurement-dependent signs and output errors can cancel.

Individual factors can violate output stabilizers, provided their combined output syndromes cancel. This codespace condition does not add declared checks. No logical measurement is required. Audit noiseless validity separately.

The distance methods default to all nonidentity Paulis on each call's support, counting a correlated two-qubit call error as one fault. This full fault set differs from the compact X/Z propagation basis in `fault_effects`.

## 7. Synthesize a protocol from the code

The same C4 definition can seed a new two-layer protocol. `ec.build_qodec` constructs gadget circuits and their declarations; it does not copy the hand-authored implementation. Inspect its instruction menu and one generated circuit, then run the same audit rather than assuming construction settled every requirement.

In [21]:
synthesized = ec.build_qodec(protocol.codes["C4"])
synthesized_gadgets = synthesized.layers[0].gadgets
display(Markdown("\n".join([
    "| Generated instruction | Circuit calls | Checks | Readouts |",
    "| --- | ---: | ---: | ---: |",
    *(f"| `{mnemonic}` | {len(gadget.circuit.calls)} | {len(gadget.checks)} | {len(gadget.readouts)} |"
      for mnemonic, gadget in sorted(synthesized_gadgets.items())),
])))
print("Generated idle circuit:")
print(synthesized_gadgets["idle"].circuit.source)
assert synthesized is not protocol
assert set(synthesized.layers[0].instruction_set.instructions) == set(synthesized_gadgets)

| Generated instruction | Circuit calls | Checks | Readouts |
| --- | ---: | ---: | ---: |
| `idle` | 16 | 4 | 0 |
| `measure_x` | 8 | 1 | 2 |
| `measure_z` | 4 | 1 | 2 |
| `prepare_x` | 24 | 3 | 0 |
| `prepare_z` | 20 | 3 | 0 |
| `x0` | 2 | 0 | 0 |
| `x1` | 2 | 0 | 0 |
| `z0` | 2 | 0 | 0 |
| `z1` | 2 | 0 | 0 |

Generated idle circuit:
R 4
H 4
CX 4 0
CX 4 1
CX 4 2
CX 4 3
H 4
R 5
H 5
CZ 5 0
CZ 5 1
CZ 5 2
CZ 5 3
H 5
M 4 5



The generated menu is not the hand-authored instruction set: for example, it has no `transversal_cx` gadget. By default, synthesis raises if one of the instructions it attempts cannot be completed and verified; it does not silently omit that instruction.

Now audit the result. The current C4 synthesizer still leaves missing incoming-frame terms in measurement readouts and incomplete output-frame relations. Successful construction is not a clean audit or a fault-tolerance certificate.

In [22]:
synthesized_report = ec.audit(synthesized)
synthesis_counts = Counter((item.severity.name, item.rule) for item in synthesized_report.diagnostics)
display(Markdown("\n".join([
    "| Severity | Rule | Count |",
    "| --- | --- | ---: |",
    *(f"| {severity} | `{rule}` | {count} |"
      for (severity, rule), count in sorted(synthesis_counts.items())),
])))
assert len(synthesized_report.errors) == 4
assert len(synthesized_report.warnings) == 8
assert all(item.source_location is None for item in synthesized_report.diagnostics)

| Severity | Rule | Count |
| --- | --- | ---: |
| ERROR | `gadget/readout-mismatch` | 4 |
| WARNING | `gadget/incomplete-output-frame` | 8 |

## 8. Save the revised design

Save the manually repaired protocol, including its remaining warnings. `Qodec.save` takes a destination directory; with `single_file=True`, the bundle is written there under `manifest_filename`. Reload that file and compare the complete declarations, not just the protocol name.

A successful round trip checks preservation, not correctness. Source locations belong to the loaded file revision and do not participate in structural equality.

In [23]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    repaired.save(directory, single_file=True)
    saved_path = Path(directory) / repaired.manifest_filename
    reloaded = qc.Qodec.load(saved_path)
    assert reloaded == repaired
    assert ec.audit(reloaded).ok

print("Complete protocol round-trips:", reloaded == repaired)

Complete protocol round-trips: True


## What we established

The distance witness changes logical information without a syndrome. Two different circuits implement the same logical X, while a third does not. Adding incoming frame signs repairs the measurement readouts, but leaves output-frame questions open. Fault propagation shows which declared checks respond to specific faults. Synthesis supplies another implementation to inspect, and qodec preserves the revised declarations on disk.

These are different kinds of evidence. A good code distance does not certify its circuits; a noiselessly correct circuit does not establish fault tolerance; and a successful save does not replace an audit.

For a next experiment, change one coupling in a fresh gadget, compare its action, and repeat the same fault cases. A fault-tolerance study would additionally need an explicit fault set, detection or correction requirements, and any decoder assumptions.